# 07 — Insulation Dataset Improvement Plan

**Purpose:** translate the completed insulation-only audit into an evidence-based data-collection and validation plan.

This notebook is **analysis and planning only**:
- It does not edit the source CSV.
- It does not create synthetic product records.
- It does not train or tune models.
- It preserves product-level grouping and provenance.
- It identifies where the current dataset can and cannot support reliable ML.

Run with the `arch_ai` kernel. Run all cells from top to bottom.

## 1. Configuration

Update `PROJECT_ROOT` only if the project has moved. The source dataset is read-only for this notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    r"D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI"
)
ML_ROOT = PROJECT_ROOT / "06_AI_Performance_Engine" / "02_ML_Engine"

DATA_FILENAME = "SpecSynth_ML_variants_v1.5.0.csv"
EXPECTED_DATA_PATH = (
    PROJECT_ROOT / "04_Product_Dataset" / "10_Master_Database" /
    "SpecSynth_Master_Database_v1.5.0" / "source_data" / "v1.5.0" /
    "ml" / DATA_FILENAME
)

# If the database folder has a slightly different version/name, search the
# project directory for the exact dataset filename instead of failing.
if EXPECTED_DATA_PATH.exists():
    DATA_PATH = EXPECTED_DATA_PATH
else:
    matches = list(PROJECT_ROOT.rglob(DATA_FILENAME)) if PROJECT_ROOT.exists() else []
    if len(matches) == 1:
        DATA_PATH = matches[0]
        print("Dataset found using project-folder search:")
        print(DATA_PATH)
    elif len(matches) > 1:
        print("More than one matching dataset was found:")
        for i, match in enumerate(matches, start=1):
            print(f"{i}. {match}")
        raise FileNotFoundError(
            "Multiple matching datasets found. Set DATA_PATH manually to the "
            "correct v1.5.0 CSV so the notebook does not select the wrong version."
        )
    else:
        raise FileNotFoundError(
            f"Could not find {DATA_FILENAME} under:\n{PROJECT_ROOT}\n\n"
            "Check PROJECT_ROOT and confirm that the v1.5.0 dataset is present."
        )

# Audit output directory created by Notebook 06
AUDIT_DIR = ML_ROOT / "04_Evaluation" / "reports" / "insulation_dataset_audit_v1"
OUTPUT_DIR = ML_ROOT / "04_Evaluation" / "reports" / "insulation_improvement_plan_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)
print("Audit directory exists:", AUDIT_DIR.exists())
print("Output directory:", OUTPUT_DIR)

Dataset found using project-folder search:
D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\04_Product_Dataset\10_Master_Database\SpecSynth_Master_Database_v1.0\source_data\v1.5.0\ml\SpecSynth_ML_variants_v1.5.0.csv
Dataset exists: True
Dataset path: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\04_Product_Dataset\10_Master_Database\SpecSynth_Master_Database_v1.0\source_data\v1.5.0\ml\SpecSynth_ML_variants_v1.5.0.csv
Audit directory exists: True
Output directory: D:\02_LEARNING\02_Masters_IAAC_AI_Architecture\10-Term_01\03-Research Studio\SpecSynth_AI\06_AI_Performance_Engine\02_ML_Engine\04_Evaluation\reports\insulation_improvement_plan_v1


**Conclusion:** confirm that the dataset and audit directory paths resolve before proceeding. If the source path differs on your machine, correct it here; do not point this notebook at a different dataset version without documenting that change.

## 2. Load source and filter insulation

The original all-material dataset remains unchanged. We create an in-memory insulation subset using the audited category-group field.

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df_all = pd.read_csv(DATA_PATH)
required = {"material_category_group", "product_id", "variant_id", "split_group_id",
            "material_family", "thermal_conductivity_W_mK"}
missing = required - set(df_all.columns)
if missing:
    raise ValueError(f"Required columns missing: {sorted(missing)}")

df = df_all.loc[df_all["material_category_group"].eq("A_thermal_insulation")].copy()
print("All-material rows:", len(df_all))
print("Insulation rows:", len(df))
print("Unique products:", df["product_id"].nunique())
print("Unique variants:", df["variant_id"].nunique())
print("Unique independent split groups:", df["split_group_id"].nunique())
print("Families:", df["material_family"].nunique())

All-material rows: 1941
Insulation rows: 1674
Unique products: 70
Unique variants: 1674
Unique independent split groups: 69
Families: 16


**Conclusion:** the row count is not the number of independent examples. For split design and generalisation, `split_group_id` and `product_id` are more meaningful than raw variant count.

## 3. Rebuild family support at product/group level

A family can appear to have many records while representing very few independent products. This table makes that distinction explicit.

In [3]:
family_support = (
    df.groupby("material_family", dropna=False)
      .agg(
          records=("variant_id", "count"),
          products=("product_id", "nunique"),
          split_groups=("split_group_id", "nunique"),
          manufacturers=("manufacturer", "nunique") if "manufacturer" in df.columns else ("product_id", "nunique")
      )
      .sort_values(["split_groups", "products", "records"], ascending=False)
)
family_support["records_per_product"] = (
    family_support["records"] / family_support["products"].replace(0, np.nan)
).round(1)
family_support["support_band"] = pd.cut(
    family_support["split_groups"],
    bins=[-1, 2, 4, 9, np.inf],
    labels=["Very low: 1–2 groups", "Low: 3–4 groups", "Moderate: 5–9 groups", "Stronger: 10+ groups"]
)
display(family_support)
family_support.to_csv(OUTPUT_DIR / "insulation_family_independent_support_v1.csv")

,records,products,split_groups,manufacturers,records_per_product,support_band
material_family,,,,,,
glass_wool,416,24,24,6,17.3,Stronger: 10+ groups
XPS,179,8,8,4,22.4,Moderate: 5–9 groups
PIR,187,7,7,4,26.7,Moderate: 5–9 groups
stone_wool,151,7,7,4,21.6,Moderate: 5–9 groups
EPS,366,6,6,3,61.0,Moderate: 5–9 groups
wood_wool,28,4,3,4,7.0,Low: 3–4 groups
wood_fibre,34,3,3,2,11.3,Low: 3–4 groups
cellulose,121,2,2,2,60.5,Very low: 1–2 groups
polyester_wool,28,2,2,1,14.0,Very low: 1–2 groups


**Conclusion:** families with only one or two independent groups cannot support a credible family-level holdout evaluation. A high variant count does not fix a low independent-product count.

## 4. Quantify manufacturer concentration

Concentrated sourcing can make a model learn manufacturer-specific patterns rather than general material behaviour.

In [4]:
manufacturer_col = "manufacturer" if "manufacturer" in df.columns else None
if manufacturer_col:
    manufacturer_support = (
        df.groupby(manufacturer_col)
          .agg(records=("variant_id", "count"),
               products=("product_id", "nunique"),
               groups=("split_group_id", "nunique"))
          .sort_values("records", ascending=False)
    )
    manufacturer_support["record_share_pct"] = (
        100 * manufacturer_support["records"] / len(df)
    ).round(2)
    display(manufacturer_support.head(15))
    top5_share = manufacturer_support.head(5)["records"].sum() / len(df) * 100
    print(f"Top-five manufacturer record share: {top5_share:.2f}%")
    manufacturer_support.to_csv(OUTPUT_DIR / "insulation_manufacturer_support_v1.csv")
else:
    print("Manufacturer field not found; inspect available columns:", df.columns.tolist())

,records,products,groups,record_share_pct
manufacturer,,,,
Hirsch France,299,4,4,17.86
Knauf Insulation SPRL,186,5,5,11.11
SOPREMA,141,4,4,8.42
ROCKWOOL France S.A.S.,135,4,4,8.06
"UNILIN BV, Division Insulation",121,3,3,7.23
URSA Insulation S.A.,88,9,9,5.26
Saint-Gobain Isover (FR),80,5,5,4.78
CAVAC Biomatériaux,62,1,1,3.70
IGLOO France Cellulose SAS,59,1,1,3.52


Top-five manufacturer record share: 52.69%


**Conclusion:** collect independent products from manufacturers that are currently under-represented. Where possible, include multiple manufacturers within the same family so that family predictions are not merely manufacturer recognition.

## 5. Identify property gaps by family

This identifies the technical fields where additional real product evidence would add the most value. Missing values must not be silently replaced with invented manufacturer declarations.

In [5]:
candidate_properties = [
    "thermal_conductivity_W_mK", "thickness_mm", "thermal_resistance_m2K_W",
    "compressive_strength_kPa", "vapour_resistance_factor_mu",
    "tensile_strength_perpendicular_kPa", "mass_per_area_kg_m2",
    "density_kg_m3", "specific_heat_J_kgK", "gwp_a1_a3_kgco2e_m2"
]
available_properties = [c for c in candidate_properties if c in df.columns]

family_coverage = df.groupby("material_family")[available_properties].agg(
    lambda s: round(100 * s.notna().mean(), 1)
)
display(family_coverage)
family_coverage.to_csv(OUTPUT_DIR / "insulation_family_property_coverage_pct_v1.csv")

overall_coverage = pd.DataFrame({
    "available": df[available_properties].notna().sum(),
    "missing": df[available_properties].isna().sum(),
    "coverage_pct": (100 * df[available_properties].notna().mean()).round(2)
}).sort_values("coverage_pct")
display(overall_coverage)
overall_coverage.to_csv(OUTPUT_DIR / "insulation_overall_property_coverage_v1.csv")

,thermal_conductivity_W_mK,thickness_mm,thermal_resistance_m2K_W,compressive_strength_kPa,vapour_resistance_factor_mu,tensile_strength_perpendicular_kPa,mass_per_area_kg_m2,density_kg_m3,specific_heat_J_kgK,gwp_a1_a3_kgco2e_m2
material_family,,,,,,,,,,
EPS,100.0,100.0,100.0,26.5,0.0,57.9,0.0,2.7,0.0,0.0
PF,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
PIR,100.0,100.0,100.0,79.1,0.0,70.6,0.0,5.9,0.0,0.0
PUR,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
XPS,93.3,100.0,96.1,100.0,73.2,73.2,6.7,6.7,0.0,6.7
aerogel,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,0.0,0.0
cellulose,100.0,100.0,100.0,0.0,48.8,0.0,100.0,0.0,0.0,0.0
cork,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,0.0
glass_wool,99.8,100.0,99.8,5.3,55.3,4.3,13.0,13.0,17.1,11.3


,available,missing,coverage_pct
gwp_a1_a3_kgco2e_m2,67,1607,4.00
specific_heat_J_kgK,115,1559,6.87
density_kg_m3,163,1511,9.74
mass_per_area_kg_m2,223,1451,13.32
tensile_strength_perpendicular_kPa,589,1085,35.19
vapour_resistance_factor_mu,603,1071,36.02
compressive_strength_kPa,607,1067,36.26
thermal_conductivity_W_mK,1614,60,96.42
thermal_resistance_m2K_W,1666,8,99.52
thickness_mm,1674,0,100.00


**Conclusion:** conductivity, thickness and thermal resistance are already comparatively complete. The largest collection gaps are likely to be density, specific heat, environmental data, mass per area and mechanical/vapour properties. Prioritise fields according to the intended ML task and evidence quality.

## 6. Conductivity target support by family and product

Before regression development, check whether each family has enough independent products and target variation. Repeated variants can share the same declared conductivity.

In [6]:
target = "thermal_conductivity_W_mK"
target_rows = df.loc[df[target].notna()].copy()

target_support = (
    target_rows.groupby("material_family")
    .agg(
        target_records=(target, "count"),
        products=("product_id", "nunique"),
        groups=("split_group_id", "nunique"),
        unique_target_values=(target, "nunique"),
        target_min=(target, "min"),
        target_median=(target, "median"),
        target_max=(target, "max"),
        target_std=(target, "std")
    )
    .sort_values(["groups", "products"], ascending=False)
)
display(target_support)
target_support.to_csv(OUTPUT_DIR / "insulation_conductivity_target_support_v1.csv")

product_target_repetition = (
    target_rows.groupby("product_id")
    .agg(records=(target, "count"),
         unique_lambda_values=(target, "nunique"),
         lambda_median=(target, "median"))
)
print("Products with more than one variant:", int((product_target_repetition["records"] > 1).sum()))
print("Products with one unique lambda across all variants:",
      int((product_target_repetition["unique_lambda_values"] == 1).sum()))
display(product_target_repetition.head(20))

,target_records,products,groups,unique_target_values,target_min,target_median,target_max,target_std
material_family,,,,,,,,
glass_wool,415,23,23,11,0.031,0.034,0.043,0.002270
PIR,187,7,7,1,0.022,0.022,0.022,0.000000
stone_wool,151,7,7,4,0.035,0.035,0.038,0.000464
EPS,366,6,6,4,0.031,0.031,0.038,0.002627
XPS,167,6,6,5,0.027,0.032,0.035,0.001175
wood_wool,28,4,3,2,0.070,0.070,0.100,0.015119
wood_fibre,34,3,3,3,0.037,0.039,0.042,0.001927
cellulose,121,2,2,2,0.040,0.041,0.041,0.000475
polyester_wool,28,2,2,2,0.034,0.034,0.039,0.002520


Products with more than one variant: 62
Products with one unique lambda across all variants: 58


,records,unique_lambda_values,lambda_median
product_id,,,
AMORIM_ICB_EXPANDED_CORKBOARD,36,1,0.039
BAUMIT_PROTHERM_EPS,10,1,0.037
CAVAC_JETFIB_OUATE_ACERMI_14_D_130_929,62,1,0.041
CELENIT_ACERMI_15_176_1071,7,1,0.100
ETEX_MULTITHERM31_ACERMI_26_009_1867,57,1,0.031
HIRSCH_CELLOCEM_ULTRA_PRIMAIRE_ECA_ACERMI_24_081_1685,77,1,0.031
HIRSCH_CELLOMUR_ULTRA_ACERMI_12_081_795,78,1,0.031
HIRSCH_MAXISOL_ECA_ACERMI_24_081_1679,87,1,0.034
HIRSCH_STISOL_3000_ECA_ACERMI_24_081_1661,57,1,0.038


**Conclusion:** product-level grouping is essential. If a product has many thickness variants but one repeated lambda declaration, those rows are not independent target observations. Keep related variants in the same split and report performance at both record and product/group level where practical.

## 7. Inspect provenance and evidence eligibility

Do not collapse verified, partially verified, calculated, or other derived values into one undifferentiated training pool. This cell reports the fields available in this dataset.

In [7]:
for col in ["data_status", "record_origin", "r_status", "evidence_quality_score", "qa_flag_count"]:
    if col in df.columns:
        print(f"\\n--- {col} ---")
        display(df[col].value_counts(dropna=False).rename("records").to_frame())

if "data_status" in df.columns:
    provenance_by_family = pd.crosstab(
        df["material_family"], df["data_status"], normalize="index"
    ).mul(100).round(1)
    display(provenance_by_family)
    provenance_by_family.to_csv(OUTPUT_DIR / "insulation_provenance_by_family_pct_v1.csv")

\n--- data_status ---


,records
data_status,
VERIFIED,1465
PARTIALLY_VERIFIED,209


\n--- record_origin ---


,records
record_origin,
new_v1.4.0,1271
new_v1.5.0,309
legacy_v1.2.0,94


\n--- r_status ---


,records
r_status,
certified_declared,1460
manufacturer_declared,97
epd_declared,88
calculated_from_manufacturer_declared_lambda,10
NaN,8
DoP declared RD,5
computed_t_over_lambda,4
calculated t/λ (NBS lists 2.00 for 80mm),1
calculated t/λ (NBS lists 2.75 for 100mm),1


\n--- evidence_quality_score ---


,records
evidence_quality_score,
3,1465
2,209


\n--- qa_flag_count ---


,records
qa_flag_count,
0,1483
1,135
2,40
3,16


data_status,PARTIALLY_VERIFIED,VERIFIED
material_family,,
EPS,2.7,97.3
PF,0.0,100.0
PIR,0.5,99.5
PUR,0.0,100.0
XPS,6.7,93.3
aerogel,100.0,0.0
cellulose,0.0,100.0
cork,0.0,100.0
glass_wool,19.0,81.0


**Conclusion:** define an explicit eligibility rule before training. A conservative first supervised experiment should use records whose target and essential input values have traceable evidence. Keep partially verified or derived records available for separate sensitivity experiments, clearly labelled and never mixed into the primary evaluation without disclosure.

## 8. Existing split: why it is insufficient for insulation-only test claims

The inherited split was designed for the full dataset. After filtering, its test set contains only a few independent product groups. Recalculate group counts directly.

In [8]:
# The source dataset uses `recommended_split` for the inherited split label.
split_candidates = ["recommended_split", "split", "dataset_split"]
split_col = next((c for c in split_candidates if c in df.columns), None)

if split_col:
    split_summary = (
        df.groupby(split_col, dropna=False)
          .agg(records=("variant_id", "count"),
               products=("product_id", "nunique"),
               groups=("split_group_id", "nunique"))
          .sort_index()
    )
    display(split_summary)
    split_summary.to_csv(OUTPUT_DIR / "insulation_inherited_split_support_v1.csv")

    # A split_group_id must belong to exactly one split label.
    group_split_counts = df.groupby("split_group_id")[split_col].nunique(dropna=False)
    crossing_groups = group_split_counts[group_split_counts > 1]
    print("Split column used:", split_col)
    print("Groups appearing in multiple splits:", len(crossing_groups))
    if len(crossing_groups):
        display(crossing_groups.to_frame("number_of_split_labels"))
else:
    print("No recognised split column found. Columns containing 'split':",
          [c for c in df.columns if "split" in c.lower()])
    raise KeyError("Could not identify the inherited split-label column.")

,records,products,groups
recommended_split,,,
test,41,3,3
train,1465,58,57
validation,168,9,9


Split column used: recommended_split
Groups appearing in multiple splits: 0


**Conclusion:** the inherited grouping prevents leakage, but the insulation-only test set is too small for a strong generalisation claim. Do not use its metrics as proof that the insulation model is production-ready.

## 9. Proposed validation strategy (planning recommendation)

**Do not randomly split rows.** Use `split_group_id` as the indivisible unit.

Recommended staged strategy:
1. **Development:** grouped cross-validation on the training/development pool (e.g. `GroupKFold` or `GroupShuffleSplit`). Keep all variants from a group together.
2. **Model selection:** compare algorithms and tune only inside development folds. Never use the final holdout to choose models or hyperparameters.
3. **Final evaluation:** create a product-group holdout with enough independent groups to be informative. Ensure it includes meaningful family and manufacturer coverage.
4. **Rare families:** mark families with very few independent groups as exploratory / unsupported for supervised family-level claims. Do not force stratification where the group counts make it impossible.
5. **Reporting:** show number of records, products and groups per split, plus family distribution. Report uncertainty and family-level results only where sample support is adequate.

For classification, first agree whether the desired target is all 16 insulation families or a narrower set of adequately represented families. This notebook does not silently merge classes.

## 10. Prioritised data improvement plan

The following priorities are recommendations derived from the audit patterns, not new verified facts about products.

In [9]:
plan = pd.DataFrame([
    ["P0", "Independent product coverage", "Add real, traceable products to families with very few independent products/groups.", "More independent products and split groups per family", "Required before broad family-classification claims"],
    ["P0", "Validation design", "Design a new group-aware development/validation/holdout strategy before tuning.", "Adequate independent groups in every split; zero group leakage", "Required before performance claims"],
    ["P1", "Manufacturer diversity", "Expand under-represented manufacturers within major families.", "Lower top-manufacturer concentration; multiple manufacturers per family", "Reduces source-specific bias"],
    ["P1", "Evidence quality", "Capture source document, declaration type, status and provenance for every new value.", "Traceable target and input evidence", "Required for trustworthy product insights"],
    ["P1", "Technical property gaps", "Prioritise density, specific heat, GWP, mass/area and mechanical/vapour properties where relevant.", "Higher family-level property coverage", "Enables additional performance tasks"],
    ["P2", "Variant dependence audit", "Check repeated targets and variant relationships at product/group level.", "Documented within-product target variation", "Prevents inflated effective sample size"],
    ["P2", "Rare-family task scope", "Agree whether to collect more products or restrict the first classifier to supported families.", "Explicit supported-family scope", "Avoids misleading rare-class metrics"]
], columns=["priority", "workstream", "recommended_action", "success_measure", "reason"])
display(plan)
plan.to_csv(OUTPUT_DIR / "insulation_dataset_improvement_plan_v1.csv", index=False)

,priority,workstream,recommended_action,success_measure,reason
0,P0,Independent product coverage,"Add real, traceable products to families with ...",More independent products and split groups per...,Required before broad family-classification cl...
1,P0,Validation design,Design a new group-aware development/validatio...,Adequate independent groups in every split; ze...,Required before performance claims
2,P1,Manufacturer diversity,Expand under-represented manufacturers within ...,Lower top-manufacturer concentration; multiple...,Reduces source-specific bias
3,P1,Evidence quality,"Capture source document, declaration type, sta...",Traceable target and input evidence,Required for trustworthy product insights
4,P1,Technical property gaps,"Prioritise density, specific heat, GWP, mass/a...",Higher family-level property coverage,Enables additional performance tasks
5,P2,Variant dependence audit,Check repeated targets and variant relationshi...,Documented within-product target variation,Prevents inflated effective sample size
6,P2,Rare-family task scope,Agree whether to collect more products or rest...,Explicit supported-family scope,Avoids misleading rare-class metrics


## 11. Suggested collection targets

Use these as **planning thresholds**, not statistical guarantees. More data may be needed depending on within-family diversity and the intended deployment scope.

- For an initial supervised family classifier, aim for at least **10 independent product groups per included family**, preferably from at least **3 manufacturers**.
- Families below that threshold should remain exploratory or be excluded from the first scoped classifier only after the project owner approves the scope.
- For conductivity regression, aim for substantially more independent product groups across the conductivity range, not merely more thickness variants.
- Reserve a final group holdout only after the dataset has enough independent groups to leave meaningful development and evaluation samples.
- Continue to report the actual group counts; do not treat these thresholds as a guarantee of model quality.

## 12. Export and final checks

The notebook exports planning tables only. The original dataset is not modified.

In [10]:
print("Exported planning files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.name)

print("\\nSource dataframe unchanged in memory:", len(df_all), "all-material rows")
print("Insulation subset:", len(df), "rows")
print("Independent insulation groups:", df["split_group_id"].nunique())
print("No model training performed in this notebook.")

Exported planning files:
- insulation_conductivity_target_support_v1.csv
- insulation_dataset_improvement_plan_v1.csv
- insulation_family_independent_support_v1.csv
- insulation_family_property_coverage_pct_v1.csv
- insulation_inherited_split_support_v1.csv
- insulation_manufacturer_support_v1.csv
- insulation_overall_property_coverage_v1.csv
- insulation_provenance_by_family_pct_v1.csv
\nSource dataframe unchanged in memory: 1941 all-material rows
Insulation subset: 1674 rows
Independent insulation groups: 69
No model training performed in this notebook.


## Final conclusion

The insulation subset is a useful starting point, but **1,674 rows should not be mistaken for 1,674 independent products**. The 70 products and 69 split groups provide a more realistic picture of the available evidence. Several families have very limited independent support, manufacturer records are concentrated, and important technical properties remain sparse.

The immediate next step is **real, provenance-backed data expansion and a redesigned group-aware validation plan**. Do not tune or package the insulation models until the intended classification scope and validation strategy are agreed. Preserve the all-material source dataset and keep any lower-confidence or derived data clearly separated from the primary evaluation.